# 04 Feature Engineering

UCI Bank Marketing Dataset — building on `01_data_understanding_and_cleaning.ipynb`, `02_eda.ipynb`, and `03_leakage_investigation.ipynb`.

This notebook creates a small number of justified, simple features consistent with the BEFORE-CALL prediction scenario. It does not encode categorical variables, scale numbers, split data, or train models — that belongs to later notebooks. `duration` is never used to build anything here, and `y` is never used to build anything here.

## 1. Load Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/bank_marketing_cleaned.csv')
df.shape

(41176, 21)

## 2. Review Previous Findings

Decisions from notebooks 01-03 that constrain what happens in this notebook:

- **`duration`** is strongly predictive but only known once the current call has happened — excluded from the BEFORE-CALL feature set (notebook 03). It is not used to build any feature here.
- **`campaign`** counts contacts this campaign *including the current one*, per its dataset definition. At the exact prediction point (immediately before this specific call), only the contacts made *before* it are actually known — notebook 03 concluded `campaign` needs a `campaign - 1` adjustment before use, not exclusion.
- **`pdays == 999`** does not reliably mean "never contacted before" — 4,110 rows have `previous > 0` (contacted before) but still carry the `999` sentinel. Any "contacted before" feature needs a more reliable source than `pdays` alone.
- **`previous`** and **`poutcome`** are internally perfectly consistent with each other (`previous == 0` exactly matches `poutcome == 'nonexistent'`).
- **`unknown`** categorical values are kept as a real category, never dropped or imputed.
- The target **`y`** must never be used to construct a feature.

## 3. Feature Engineering Principles

Every feature considered in this notebook is checked against one question:

> **Could this information be known immediately before the planned customer contact?**

And every feature that gets created satisfies:

1. It solves a specific problem found in notebooks 01-03 (not "more columns for their own sake").
2. It only uses information available at prediction time.
3. It does not use `duration` or `y`.
4. It's simple enough to explain in one sentence.

Categorical encoding, scaling, and building the final model-ready feature matrix are left for the preprocessing/pipeline notebook — this notebook only creates readable, meaningful columns.

## 4. Previous Contact Features

**Goal:** a simple flag indicating whether this client has any prior campaign contact history. The naive definition `pdays != 999` is unreliable (see above), so the three related columns are checked against each other first.

In [2]:
consistency_check = pd.crosstab(df['previous'] == 0, df['poutcome'])
consistency_check

poutcome,failure,nonexistent,success
previous,,,
False,4252,0,1373
True,0,35551,0


In [3]:
previous_flag = df['previous'] > 0
pdays_flag = df['pdays'] != 999

agreement = (previous_flag == pdays_flag).mean()
print(f'previous > 0 agrees with pdays != 999 for {agreement:.1%} of rows')

previous > 0 agrees with pdays != 999 for 90.0% of rows


`previous == 0` lines up **exactly** with `poutcome == 'nonexistent'` — no exceptions. `previous > 0` only agrees with `pdays != 999` for 90.0% of rows, confirming the mismatch found in notebook 02: `pdays` misses roughly 10% of clients who were genuinely contacted before.

**Definition used:** `was_previously_contacted = previous > 0`. This is backed by `previous` and `poutcome` agreeing on every single row, so the feature is reliable regardless of `pdays`'s recording gaps. All three columns describe history from *before the current campaign*, so this is available at prediction time without question.

In [4]:
df['was_previously_contacted'] = (df['previous'] > 0).astype(int)
df['was_previously_contacted'].value_counts()

was_previously_contacted
0    35551
1     5625
Name: count, dtype: int64

In [5]:
# should match poutcome != 'nonexistent' exactly
(df['was_previously_contacted'] == (df['poutcome'] != 'nonexistent').astype(int)).all()

np.True_

## 5. Age-Based Features

**Goal:** check whether age has a relationship with subscription that a single continuous number represents poorly, before deciding whether grouping is justified.

In [6]:
age_bins = [17, 25, 35, 45, 55, 65, 99]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '66+']

age_group_preview = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)
rate_by_age_group = df.groupby(age_group_preview, observed=True)['y'].apply(lambda s: (s == 'yes').mean() * 100).round(2)
count_by_age_group = df.groupby(age_group_preview, observed=True).size()
pd.DataFrame({'count': count_by_age_group, 'subscription_rate_pct': rate_by_age_group})

,count,subscription_rate_pct
age,,
18-25,1665,20.96
26-35,14844,11.72
36-45,12839,8.51
46-55,8247,8.69
56-65,2963,15.22
66+,618,46.93


This is a **U-shaped** pattern, not a straight-line trend: the rate is highest for the youngest group (18-25: 20.9%), drops to its lowest in middle age (36-45: 8.5%, 46-55: 8.7%), then rises again for older clients (56-65: 15.2%, 66+: 46.9%). This lines up with notebook 02's `job` finding, where `student` and `retired` — disproportionately young and old clients — had the two highest subscription rates of any job category.

This is the justification for grouping: a single continuous `age` number can represent a *consistent* trend well, but not a pattern that goes down and then back up. `age_group` is added **alongside** `age`, not as a replacement — the raw continuous value is still useful (e.g. for tree-based models, which can find this pattern on their own), and dropping it would throw away information for no benefit.

In [7]:
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)
df['age_group'].isnull().sum()

np.int64(0)

## 6. Campaign Features

Notebook 03's conclusion: `campaign` counts contacts this campaign *including the current one*, but at the moment right before this specific call, only the contacts made **before** it are actually known. The fix is a simple, fixed offset: `campaign - 1`.

In [8]:
df['contacts_before_this_call'] = df['campaign'] - 1
df['contacts_before_this_call'].describe()

count    41176.000000
mean         1.567879
std          2.770318
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         55.000000
Name: contacts_before_this_call, dtype: float64

Minimum is 0 (a client's first-ever contact this campaign has zero prior contacts, which is correct), and no negative values appear. This column represents exactly what's known before dialing — the original `campaign` column is left unchanged alongside it for reference.

**Is a low/medium/high contact-intensity grouping also justified?** Notebook 02 already found a clean, monotonically decreasing subscription rate as contact count rises (13.0% at 1 contact down to 4.6% at 7+). That's a simple, consistent trend — exactly the case where binning adds complexity without adding information a continuous number doesn't already capture. **No additional campaign-intensity feature is created.**

## 7. Previous Campaign Features

`previous`, `poutcome`, and `pdays` were already investigated in Section 4, which produced `was_previously_contacted`. One more idea is worth checking explicitly before moving on: should the `previous > 0` / `pdays == 999` mismatch (4,110 rows) get its own flag?

In [9]:
mismatch_count = ((df['previous'] > 0) & (df['pdays'] == 999)).sum()
mismatch_count

np.int64(4110)

**Considered and rejected.** A dedicated flag for this mismatch would describe a gap in how `pdays` happened to be recorded, not a real property of the customer or campaign — it's specific to one historical data-collection quirk rather than a generalizable pattern. `was_previously_contacted` already captures the reliable version of this information (Section 4), and `pdays` itself is left unchanged so that later preprocessing can decide how to handle its `999` sentinel directly. Adding a second, narrower flag on top would be redundant with information the model can already get from `pdays` and `was_previously_contacted` together.

**No additional feature is created in this section.**

## 8. Other Justified Features

### Categorical grouping

Checking every categorical column's smallest category, to see whether any is small enough to be unreliable rather than just uncommon.

In [10]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

for col in categorical_cols:
    counts = df[col].value_counts()
    smallest_category = counts.idxmin()
    smallest_count = counts.min()
    print(f'{col}: smallest category is "{smallest_category}" with {smallest_count} rows ({smallest_count / len(df) * 100:.2f}%)')

job: smallest category is "unknown" with 330 rows (0.80%)
marital: smallest category is "unknown" with 80 rows (0.19%)
education: smallest category is "illiterate" with 18 rows (0.04%)
default: smallest category is "yes" with 3 rows (0.01%)
housing: smallest category is "unknown" with 990 rows (2.40%)
loan: smallest category is "unknown" with 990 rows (2.40%)
contact: smallest category is "telephone" with 15041 rows (36.53%)
month: smallest category is "dec" with 182 rows (0.44%)
day_of_week: smallest category is "fri" with 7826 rows (19.01%)
poutcome: smallest category is "success" with 1373 rows (3.33%)


Most columns' smallest category is still a few hundred rows or more — small, but large enough to estimate reliably. Two columns stand out:

- `education`'s `illiterate` category has only 18 rows (0.04%) — an order of magnitude smaller than every other column's smallest category. `illiterate` sits at the low end of an ordinal schooling scale, directly adjacent to `basic.4y`, so grouping it there is a meaningful merge, not an arbitrary one.
- `default`'s `yes` category has only 3 rows (0.01%) — even smaller. But `default` isn't ordinal: `yes`, `no`, and `unknown` are three distinct, non-adjacent states, and merging `yes` into either of the others would misrepresent a client's actual known credit-default status. There's no group to merge it into without changing its meaning, so it's left as-is.

`marital`'s `unknown` (80 rows) is also relatively small but not in the same range as `illiterate` or `default == 'yes'`, and has no natural merge target either — left unchanged.

**Decision:** group only `education`'s `illiterate` into `basic.4y`, in a new column so the original `education` is preserved.

In [11]:
df['education_grouped'] = df['education'].replace({'illiterate': 'basic.4y'})
df['education_grouped'].value_counts()

education_grouped
university.degree      12164
high.school             9512
basic.9y                6045
professional.course     5240
basic.4y                4194
basic.6y                2291
unknown                 1730
Name: count, dtype: int64

In [12]:
# basic.4y count should have grown by exactly the 18 rows that used to be illiterate
print(df['education'].value_counts()['basic.4y'])
print(df['education_grouped'].value_counts()['basic.4y'])
print('illiterate' in df['education_grouped'].unique())

4176
4194
False


### Macroeconomic features

Notebook 02 found `emp.var.rate`, `cons.price.idx`, `euribor3m`, and `nr.employed` strongly inter-correlated (pairwise 0.69-0.97).

In [13]:
economic_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
df[economic_cols].corr().round(2)

,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
emp.var.rate,1.00,0.78,0.20,0.97,0.91
cons.price.idx,0.78,1.00,0.06,0.69,0.52
cons.conf.idx,0.20,0.06,1.00,0.28,0.10
euribor3m,0.97,0.69,0.28,1.00,0.95
nr.employed,0.91,0.52,0.10,0.95,1.00


Combining features that are already this strongly correlated (e.g. averaging or taking a ratio) would not add new information — it would mostly restate what's already shared between them, while making the result harder to interpret than the original indicators. This kind of multicollinearity is better handled at the modeling stage (feature selection, regularization, or using a model family that isn't sensitive to it) than by manually combining variables here.

**No additional feature engineering is justified for the economic indicators. They are left unchanged.**

## 9. Feature Validation

Sanity checks on all four new columns together.

In [14]:
new_columns = ['was_previously_contacted', 'age_group', 'contacts_before_this_call', 'education_grouped']
df[new_columns].isnull().sum()

was_previously_contacted     0
age_group                    0
contacts_before_this_call    0
education_grouped            0
dtype: int64

In [15]:
print('was_previously_contacted unique values:', sorted(df['was_previously_contacted'].unique()))
print('contacts_before_this_call min:', df['contacts_before_this_call'].min())
print('age_group categories:', df['age_group'].cat.categories.tolist())
print('education_grouped categories:', sorted(df['education_grouped'].unique()))

was_previously_contacted unique values: [np.int64(0), np.int64(1)]
contacts_before_this_call min: 0
age_group categories: ['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
education_grouped categories: ['basic.4y', 'basic.6y', 'basic.9y', 'high.school', 'professional.course', 'university.degree', 'unknown']


In [16]:
expected_rate = 1 - (df['poutcome'] == 'nonexistent').mean()
actual_rate = df['was_previously_contacted'].mean()
print(f'expected: {expected_rate:.4f}, actual: {actual_rate:.4f}')

expected: 0.1366, actual: 0.1366


No missing values in any new column, no negative contact counts, `was_previously_contacted` is strictly 0/1, and its overall rate matches the proportion of clients with a recorded prior campaign outcome exactly — everything behaves as designed.

## 10. Before vs After Feature Set

In [17]:
print('Before feature engineering:')
print('shape:', pd.read_csv('../data/processed/bank_marketing_cleaned.csv').shape)

Before feature engineering:
shape: (41176, 21)


In [18]:
print('After feature engineering:')
print('shape:', df.shape)
print('new columns:', new_columns)

After feature engineering:
shape: (41176, 25)
new columns: ['was_previously_contacted', 'age_group', 'contacts_before_this_call', 'education_grouped']


In [19]:
df[new_columns].dtypes

was_previously_contacted        int64
age_group                    category
contacts_before_this_call       int64
education_grouped              object
dtype: object

## 11. Save Engineered Dataset

`duration` is kept in this saved file rather than deleted — nothing is destroyed here, consistent with every prior notebook. What changes is that the BEFORE-CALL feature set is now explicitly defined below, and it excludes `duration` by name, so notebook 05 has no ambiguity about which columns to model with.

In [20]:
before_call_features = [col for col in df.columns if col not in ['y', 'duration']]
print(f'{len(before_call_features)} columns available for the BEFORE-CALL model:')
before_call_features

23 columns available for the BEFORE-CALL model:


['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'was_previously_contacted',
 'age_group',
 'contacts_before_this_call',
 'education_grouped']

In [21]:
df.to_csv('../data/processed/bank_marketing_features.csv', index=False)

## Feature Engineering Summary

### Features created

| Feature | Definition | Why | Available at prediction time? |
|---|---|---|---|
| `was_previously_contacted` | `previous > 0` (1/0) | Reliable "contacted before" flag - `previous` agrees with `poutcome` on every row, unlike the unreliable `pdays != 999` | Yes - describes campaigns that concluded before the current one |
| `age_group` | `age` binned into 18-25 / 26-35 / 36-45 / 46-55 / 56-65 / 66+ | Subscription rate is U-shaped across age (high young, low middle, high old), a pattern a single continuous number represents poorly | Yes - age is known before any contact |
| `contacts_before_this_call` | `campaign - 1` | `campaign` as stored includes the current, not-yet-made contact; this adjustment reflects only contacts truly known before dialing (notebook 03 decision) | Yes, by construction - it excludes the current contact |
| `education_grouped` | `education` with `illiterate` (18 rows, 0.04%) merged into `basic.4y` | `illiterate` was too small a sample to estimate reliably on its own; the two categories are adjacent on the schooling scale | Yes - education is known before any contact |

### Features considered and rejected

- **A `pdays == 999` mismatch flag** — would describe a data-recording quirk (Section 7), not a generalizable pattern, and would duplicate information `was_previously_contacted` and raw `pdays` already provide together.
- **Campaign contact-intensity bins (low/medium/high)** — the relationship between contact count and subscription is already a clean, monotonic trend (Section 6); binning would add complexity without adding information.
- **Grouping `default`'s rare `yes` category (3 rows)** — even smaller than `illiterate`, but `yes`/`no`/`unknown` are not ordinal, and there's no other category it could be merged into without misrepresenting a client's actual known credit-default status (Section 8).
- **Combining the macroeconomic indicators** — they are already strongly correlated (0.69-0.97); combining them would restate shared information rather than add new information, and multicollinearity is better addressed during modeling than by manual combination here (Section 8).

### Final counts and output

- 4 features created, 0 features removed.
- Output saved to `data/processed/bank_marketing_features.csv`.
- `data/processed/bank_marketing_cleaned.csv` and `data/raw/bank-additional-full.csv` are both unchanged.

### Explicit confirmations

- `duration` is excluded from the BEFORE-CALL feature set (defined explicitly in Section 11 as `before_call_features`), because it is only known once the current call has already happened. It remains in the saved CSV for transparency, but is not part of the modeling feature list.
- No engineered feature in this notebook uses `y` anywhere in its construction.
- No engineered feature uses `duration` or any other information that would only exist during or after the current planned contact.